In [1]:
# Procesamiento de datos y estimacion del potencial energetico renovable.
# v4: PVGIS como fuente primaria (24 provincias), precipitacion NASA POWER mensual,
#     runoff_coeff fijo (sin leakage), viento Hellman 10m-100m.

import pandas as pd
import numpy as np
import pvlib
import pytz
import json
import os
import requests
import time
from collections import defaultdict

if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

# Carga de datos de irradiacion solar desde PVGIS (todas las 24 provincias).
pvgis_df = pd.read_csv('../data/processed/pvgis_data_all_provinces.csv')
pvgis_df.rename(columns={'time(UTC)': 'time'}, inplace=True)
pvgis_df['time'] = pd.to_datetime(pvgis_df['time'])
pvgis_df.set_index('time', inplace=True)
print(f"[OK] PVGIS: {len(pvgis_df)} filas, {pvgis_df['provincia'].nunique()} provincias")

# ============================================================
# ZONAS CLIMATICAS (nombres con tildes, coinciden con PVGIS)
# ============================================================
ZONAS_CLIMATICAS = {
    'Costa':    ['Esmeraldas', 'Manabí', 'Santa Elena', 'Guayas', 'El Oro',
                 'Los Ríos', 'Santo Domingo de los Tsáchilas'],
    'Sierra':   ['Carchi', 'Imbabura', 'Pichincha', 'Cotopaxi', 'Tungurahua',
                 'Bolívar', 'Chimborazo', 'Cañar', 'Azuay', 'Loja'],
    'Amazonia': ['Sucumbíos', 'Napo', 'Orellana', 'Pastaza',
                 'Morona Santiago', 'Zamora Chinchipe'],
    'Insular':  ['Galápagos']
}

def get_zona(provincia):
    for zona, provs in ZONAS_CLIMATICAS.items():
        if provincia in provs:
            return zona
    return 'Costa'

def get_hellman_alpha(provincia):
    return {'Costa': 0.14, 'Sierra': 0.25, 'Amazonia': 0.20, 'Insular': 0.10}.get(
        get_zona(provincia), 0.14)

# ============================================================
# DESCARGA DE PRECIPITACION NASA POWER (mensual, 2015-2023)
# Fuente independiente de los features -> sin data leakage
# ============================================================
PRCP_CACHE = '../data/raw/nasa_cache/monthly_prcp.json'
os.makedirs('../data/raw/nasa_cache', exist_ok=True)

def download_monthly_prcp(lat, lon):
    url = 'https://power.larc.nasa.gov/api/temporal/monthly/point'
    params = {'start': '2015', 'end': '2023', 'latitude': lat, 'longitude': lon,
              'community': 'RE', 'parameters': 'PRECTOTCORR', 'format': 'JSON'}
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        raw = resp.json()['properties']['parameter']['PRECTOTCORR']
        monthly = defaultdict(list)
        for key, val in raw.items():
            m = int(key[4:6])
            if 1 <= m <= 12 and val > 0:
                monthly[m].append(val)
        return {str(m): round(sum(v)/len(v)*30, 2) for m, v in monthly.items()}
    except Exception as e:
        print(f'  [AVISO] NASA POWER fallo ({lat},{lon}): {e}')
        return {}

if os.path.exists(PRCP_CACHE) and os.path.getsize(PRCP_CACHE) > 100:
    with open(PRCP_CACHE) as f:
        prcp_by_province = json.load(f)
    print(f'[OK] Precipitacion desde cache: {len(prcp_by_province)} provincias')
else:
    prcp_by_province = {}
    print('Descargando precipitacion NASA POWER...')
    for prov in sorted(pvgis_df['provincia'].unique()):
        sub = pvgis_df[pvgis_df['provincia'] == prov]
        lat, lon = sub['latitude'].iloc[0], sub['longitude'].iloc[0]
        prcp_by_province[prov] = download_monthly_prcp(lat, lon)
        print(f"  {prov}: ene={prcp_by_province[prov].get('1','?')} mm/mes")
        time.sleep(0.5)
    with open(PRCP_CACHE, 'w') as f:
        json.dump(prcp_by_province, f, indent=2, ensure_ascii=False)
    print(f'[OK] Cache guardado en {PRCP_CACHE}')

# ============================================================
# FUNCIONES DE CALCULO DE POTENCIAL
# ============================================================

def calculate_solar_potential(pvgis_monthly):
    if pvgis_monthly.empty:
        return 0.0
    lat = pvgis_monthly['latitude'].iloc[0]
    lon = pvgis_monthly['longitude'].iloc[0]
    tilt = max(abs(lat) * 0.9, 5.0)
    azimuth = 0 if lat < 0 else 180
    tz = pytz.timezone('America/Guayaquil')
    location = pvlib.location.Location(latitude=lat, longitude=lon, tz=tz)
    idx = pvgis_monthly.index
    idx_local = (idx.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')
                 if idx.tz is None else idx.tz_convert(tz)).dropna()
    if idx_local.empty:
        return 0.0
    ml = pvgis_monthly.loc[idx_local]
    solpos = location.get_solarposition(ml.index)
    poa = pvlib.irradiance.get_total_irradiance(
        surface_tilt=tilt, surface_azimuth=azimuth,
        solar_zenith=solpos['apparent_zenith'], solar_azimuth=solpos['azimuth'],
        dni=ml['dni'], ghi=ml['ghi'], dhi=ml['dhi']
    )
    return round((poa['poa_global'].sum() / 1000) * 0.15, 4)

def power_curve(ws):
    if ws < 3 or ws > 25: return 0.0
    return (ws / 12) ** 3 * 2.0 if ws <= 12 else 2.0

def calculate_wind_potential(pvgis_monthly, provincia=''):
    # PVGIS entrega viento a 10m -> Hellman 10m-100m
    if pvgis_monthly.empty:
        return 0.0
    wind_10m = pvgis_monthly['wind_speed']
    alpha = get_hellman_alpha(provincia)
    wind_100m = wind_10m * (100 / 10) ** alpha
    return round(wind_100m.apply(power_curve).sum(), 4)

def calculate_hydro_potential(prcp_mm_mes, area_km2=500, height_m=200):
    # runoff_coeff FIJO: el target no depende de humidity_avg ni temp_avg
    runoff_coeff = 0.35
    precip_m = prcp_mm_mes / 1000
    area_m2 = area_km2 * 1e6
    dias_mes = 30
    segundos_mes = dias_mes * 24 * 3600
    volumen_m3 = precip_m * area_m2 * runoff_coeff
    caudal_m3s = volumen_m3 / segundos_mes
    potencia_mw = (1000 * 9.81 * caudal_m3s * height_m * 0.85) / 1e6
    return round(potencia_mw * (dias_mes * 24), 4)

# ============================================================
# LOOP PRINCIPAL: 24 provincias x 12 meses = 288 registros
# Features desde PVGIS: temp_air, wind_speed, relative_humidity
# ============================================================
final_dataset = []
print('\nProcesando potenciales por provincia y mes...')

for provincia in sorted(pvgis_df['provincia'].unique()):
    prov_data = pvgis_df[pvgis_df['provincia'] == provincia]
    lat = prov_data['latitude'].iloc[0]
    lon = prov_data['longitude'].iloc[0]
    prcp_mensual = prcp_by_province.get(provincia, {})
    print(f'  {provincia}...')

    for month in range(1, 13):
        pm = prov_data[prov_data.index.month == month]
        if pm.empty:
            continue
        prcp_mm = float(prcp_mensual.get(str(month), 0.0))
        row = {
            'provincia':               provincia,
            'latitude':                lat,
            'longitude':               lon,
            'month':                   month,
            'temp_avg':                round(pm['temp_air'].mean(), 4),
            'wind_speed_avg':          round(pm['wind_speed'].mean(), 4),
            'humidity_avg':            round(pm['relative_humidity'].mean(), 4),
            'target_solar_kwh_per_m2': calculate_solar_potential(pm),
            'target_wind_mwh':         calculate_wind_potential(pm, provincia),
            'target_hydro_mwh':        calculate_hydro_potential(prcp_mm),
        }
        final_dataset.append(row)

final_df = pd.DataFrame(final_dataset)

# ============================================================
# IMPUTACION POR ZONA CLIMATICA
# ============================================================
final_df['zona_climatica'] = final_df['provincia'].apply(get_zona)
final_df['humidity_avg'] = final_df.groupby('zona_climatica')['humidity_avg'].transform(
    lambda x: x.fillna(x.median()))
costa_median = final_df[final_df['zona_climatica'] == 'Costa']['humidity_avg'].median()
final_df['humidity_avg'] = final_df['humidity_avg'].fillna(costa_median)
final_df.drop(columns=['zona_climatica'], inplace=True)

print(f'\n[OK] Dataset: {len(final_df)} filas, {final_df["provincia"].nunique()} provincias')
print(f'     NaN total: {final_df.isna().sum().sum()}')
print(final_df[['target_solar_kwh_per_m2','target_wind_mwh','target_hydro_mwh']].describe().round(3))

final_df.to_csv('../data/processed/final_dataset_ecuador.csv', index=False)
print("\n[OK] Dataset guardado en '../data/processed/final_dataset_ecuador.csv'")


[OK] PVGIS: 210240 filas, 24 provincias
[OK] Precipitacion desde cache: 24 provincias

Procesando potenciales por provincia y mes...
  Azuay...
  Bolívar...
  Carchi...
  Cañar...
  Chimborazo...
  Cotopaxi...
  El Oro...
  Esmeraldas...
  Galápagos...
  Guayas...
  Imbabura...
  Loja...
  Los Ríos...
  Manabí...
  Morona Santiago...
  Napo...
  Orellana...
  Pastaza...
  Pichincha...
  Santa Elena...
  Santo Domingo de los Tsáchilas...
  Sucumbíos...
  Tungurahua...
  Zamora Chinchipe...

[OK] Dataset: 288 filas, 24 provincias
     NaN total: 0
       target_solar_kwh_per_m2  target_wind_mwh  target_hydro_mwh
count                  288.000          288.000           288.000
mean                    20.726           22.589          9412.527
std                      3.417           60.084          6428.176
min                     13.478            0.000            62.423
25%                     18.217            0.243          3548.379
50%                     20.383            3.756     

In [2]:
# === VERIFICACION ANTI-LEAKAGE (diagnostico, no parte del pipeline) ===
import pandas as pd
from sklearn.linear_model import LinearRegression

df = pd.read_csv('../data/processed/final_dataset_ecuador.csv')
print(f'Dataset: {len(df)} filas, {df["provincia"].nunique()} provincias')
print(f'NaN total: {df.isna().sum().sum()}')

features_h = ['latitude', 'longitude', 'month', 'temp_avg', 'wind_speed_avg']
X = df[features_h].values
y = df['target_hydro_mwh'].values
r2 = LinearRegression().fit(X, y).score(X, y)
print(f'\nR2 LR hidrico (5 features): {r2:.4f}  -> debe ser <0.99')
assert r2 < 0.99, f'LEAKAGE DETECTADO: R2={r2:.4f}'
print('[OK] Sin data leakage en target hidrico')

print('\nDescripcion targets:')
print(df[['target_solar_kwh_per_m2','target_wind_mwh','target_hydro_mwh']].describe().round(3))


Dataset: 288 filas, 24 provincias
NaN total: 0

R2 LR hidrico (5 features): 0.4187  -> debe ser <0.99
[OK] Sin data leakage en target hidrico

Descripcion targets:
       target_solar_kwh_per_m2  target_wind_mwh  target_hydro_mwh
count                  288.000          288.000           288.000
mean                    20.726           22.589          9412.527
std                      3.417           60.084          6428.176
min                     13.478            0.000            62.423
25%                     18.217            0.243          3548.379
50%                     20.383            3.756          8651.252
75%                     23.112           16.115         13851.204
max                     31.947          450.199         27390.699
